In [ ]:
import pandas as pd
import json
from collections import Counter

df_test = pd.read_csv('test.csv', index_col=0)
df_keyword = pd.read_csv('TestSet_ExtractKeyword_Gemma3-27B.csv')

all_keywords = set()
problem_to_keywords = {}
list_keyword_refined = [[] for _ in range(len(df_test))]
print(list_keyword_refined)
for i in range(len(df_test)):
    list_keyword = []

    for r in range(4):
        for s in range(2):
            try:
                if str(df_keyword.loc[i, f'keyword_{r}_{s}']).count('[') == 1:
                    data = str(df_keyword.loc[i, f'keyword_{r}_{s}']).split('[')[1].split(']')[0].split('"')
                    if len(data) % 2 == 1:
                        for j in range(1, len(data), 2):
                            if '(' in data[j]:
                                list_keyword.append(data[j].split('(')[0].strip().replace(" ", ""))
                            else:
                                list_keyword.append(data[j].replace(" ", ""))
            except:
                pass
    
    if not list_keyword:
        continue

    counter = Counter(list_keyword)
    for word, count in counter.items():
        if count > 6 and len(word) > 1 and '○' not in word:
            list_keyword_refined[i].append(word)
            all_keywords.add(word)
    print(i, list_keyword_refined[i])
    problem_to_keywords[i] = list_keyword_refined[i]

print(len(all_keywords))

idx = 0
idx_to_word, word_to_idx = {}, {}
for word in all_keywords:
    idx_to_word[idx] = word
    word_to_idx[word] = idx
    idx += 1

with open("idx_to_word.json", "w") as f:
    json.dump(idx_to_word, f, ensure_ascii=False, indent=4)
with open("word_to_idx.json", "w") as f:
    json.dump(word_to_idx, f, ensure_ascii=False, indent=4)

for problem, keywords in problem_to_keywords.items():
    problem_to_keywords[problem] = [word_to_idx[word] for word in keywords]
with open("problem_to_keywords.json", "w") as f:
    json.dump(problem_to_keywords, f, ensure_ascii=False, indent=4)